In [0]:

# ============================================================
# CREDIT CARD DEFAULT PREDICTION — XGBOOST OPTIMIZATION
# ============================================================

%pip install xgboost

# ------------------------------------------------------------
# 1. LOAD AND PREPARE DATA
# ------------------------------------------------------------

import pandas as pd
import numpy as np

df = spark.table("CCC_FEATURES")
pdf = df.toPandas()

pdf["NO_POSITIVE_BALANCE"] = (
    pdf["AVG_BILL_AMT"] <= 0
).astype(int)

pdf["PAYMENT_TO_BALANCE"] = (
    pdf["PAYMENT_TO_BALANCE"].fillna(0)
)

features = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "AVG_UTILIZATION",
    "PAYMENT_TO_BALANCE",
    "NO_POSITIVE_BALANCE",
    "DELINQUENT_MONTHS",
    "MAX_DELINQUENCY",
    "BALANCE_CHANGE"
]

X = pdf[features]
y = pdf["dpnm"]


# ------------------------------------------------------------
# 2. CREATE SAME TRAIN/TEST SPLIT
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))


# ------------------------------------------------------------
# 3. DEFINE XGBOOST MODEL
# ------------------------------------------------------------

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# 4. DEFINE HYPERPARAMETER SEARCH SPACE
# ------------------------------------------------------------

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2]
}


# ------------------------------------------------------------
# 5. RANDOMIZED CROSS-VALIDATION SEARCH
# ------------------------------------------------------------

from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print(
    "\nBest Cross-Validation ROC-AUC:",
    round(random_search.best_score_, 4)
)


# ------------------------------------------------------------
# 6. EVALUATE OPTIMIZED MODEL ON TEST DATA
# ------------------------------------------------------------

best_xgb = random_search.best_estimator_

optimized_pred = best_xgb.predict(X_test)
optimized_prob = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, optimized_pred)
precision = precision_score(y_test, optimized_pred)
recall = recall_score(y_test, optimized_pred)
f1 = f1_score(y_test, optimized_pred)
auc = roc_auc_score(y_test, optimized_prob)

print("\nOptimized XGBoost Results")
print("---------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

cm = confusion_matrix(y_test, optimized_pred)

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 7. COMPARE ALL MODELS
# ------------------------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Baseline XGBoost",
        "Optimized XGBoost"
    ],
    "Accuracy": [
        0.8035,
        0.8112,
        accuracy
    ],
    "Precision": [
        0.6171,
        0.6216,
        precision
    ],
    "Recall": [
        0.2939,
        0.3738,
        recall
    ],
    "F1": [
        0.3982,
        0.4668,
        f1
    ],
    "ROC_AUC": [
        0.7433,
        0.7599,
        auc
    ]
})

display(results)


## Optimization Results

#Hyperparameter optimization improved the XGBoost model's overall ability to
#differentiate between customers who defaulted and those who did not.

#ROC-AUC increased from **0.7599 to 0.7822**, while precision increased from
#**62.16% to 66.90%** and overall accuracy increased from **81.12% to 81.92%**.

#Recall decreased slightly from **37.38% to 36.10%** at the standard 0.50
#classification threshold. This illustrates the tradeoff between precision
#and recall when converting predicted probabilities into binary
#classifications.

#Because the project's business objective ultimately involves ranking
#customers by default risk for targeted intervention, ROC-AUC and the quality
#of predicted risk rankings are particularly important. The optimized model
#will therefore be used for subsequent model interpretation and customer
#prioritization analysis.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 57.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 72.3 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Training observations: 24000
Testing observations: 6000
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Best Parameters:
{'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.7}

Best Cross-Validation ROC-AUC: 0.7882

Optimized XGBoost Results
---------------------------
Accuracy:  0.8192
Precision: 0.6690
Recall:    0.3610
F1 Score:  0.4689
ROC-AUC:   0.7822

Confusion Matrix:
[[4436  237]
 [ 848  479]]


Model,Accuracy,Precision,Recall,F1,ROC_AUC
Logistic Regression,0.8035,0.6171,0.2939,0.3982,0.7433
Baseline XGBoost,0.8112,0.6216,0.3738,0.4668,0.7599
Optimized XGBoost,0.8191666666666667,0.6689944134078212,0.36096458176337604,0.468918257464513,0.7822156043689872
